# Прогнозирование цены бриллиантов

Задача проекта: предсказывать стоимость бриллианта по его характеристикам и сравнить обычную линейную регрессию с L1 и L2 регуляризацией.

## Импорты и загрузка данных

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error

In [ ]:
df = pd.read_csv('diamonds.csv')
df.head(5)

## Первичная проверка данных

In [ ]:
df.isna().sum()

In [ ]:
df = df.drop('Unnamed: 0', axis=1)
df.head()

В датасете нет пропусков. Технический столбец `Unnamed: 0` удаляется, так как не содержит информации о бриллианте.

## Корреляция с ценой

In [ ]:
corr_matrix = df.corr(numeric_only=True)
corr_matrix['price'].sort_values(ascending=False)

Наиболее сильная линейная связь с `price` наблюдается у `carat`, а также у геометрических признаков `x`, `y` и `z`.

## Кодирование, split и масштабирование

In [ ]:
df = pd.get_dummies(df, drop_first=True)
df.head()

In [ ]:
np.random.seed(67)
X = df.drop('price',axis=1)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=67, test_size=0.3)

print(X_train.shape)
print(X_test.shape)
print()
print(y_train.shape)
print(y_test.shape)
print()
print(X_train.shape[0] / df.shape[0]*100)
print(X_test.shape[0] / df.shape[0]*100)


In [ ]:
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[['carat','depth','table','x' ,'y','z']] = scaler.fit_transform(X_train[['carat','depth','table','x' ,'y','z']])
X_test_scaled[['carat','depth','table','x' ,'y','z']] = scaler.transform(X_test[['carat','depth','table','x' ,'y','z']])
X_train_scaled

Категориальные признаки кодируются через One-Hot Encoding. Числовые признаки масштабируются по параметрам, рассчитанным только на train.

## Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

lr = LinearRegression().fit(X_train_scaled, y_train)
print(f"Train: {mean_squared_error(y_train, lr.predict(X_train_scaled))}")
print(f"Test: {mean_squared_error(y_test, lr.predict(X_test_scaled))}")

In [ ]:
coefs = pd.Series(lr.coef_,index =X_train.columns)
print('Все оэффициенты')
print(coefs)
print()
print('Коэффициенты при вещественных признаках ')
print(coefs[['carat','depth','table','x' ,'y','z']])



После стандартизации величины коэффициентов можно сравнивать между собой, но они уже не интерпретируются в исходных единицах признаков.

## Ridge и Lasso

In [ ]:
from sklearn.linear_model import Lasso, Ridge

lasso = Lasso(10).fit(X_train_scaled, y_train)
print("Lasso")
print(f"Train: {mean_squared_error(y_train, lasso.predict(X_train_scaled))}")
print(f"Test: {mean_squared_error(y_test, lasso.predict(X_test_scaled))}")

ridge = Ridge(10).fit(X_train_scaled, y_train)
print("\nRidge")
print(f"Train: {mean_squared_error(y_train, ridge.predict(X_train_scaled))}")
print(f"Test: {mean_squared_error(y_test, ridge.predict(X_test_scaled))}")

In [ ]:
print('Коэффы обычной лин регрессии')
print(lr.coef_)
print()
print('Коэффы Lasso')
print(lasso.coef_)
print()
print('Коэффы Ridge')
print(ridge.coef_)

Регуляризация уменьшает абсолютные значения коэффициентов. Lasso дополнительно зануляет часть весов, а Ridge плавно сжимает их.